## Problem Statement

#### Suppose you are a Data Scientsist at a retail store like Walmart/DMart

- The company wants to increase sales and customer satisfaction through their purchase journey
- Your task is to use data to determine the products that are often purchased together:
 - Retailers can optimize product placement
 - Offer special deals and create new product bundles to encourage further sales of these combinations

<br>


<br>

> **Q. How many products are we dealing with here?**

At huge super stores like Walmart, we will have products across many categories: Daliy essentials, Food products (like Milk, Butter, jam , bread, etc), Beauty Products, Toys, etc.

Even then, we're looking at a **few hundreds** of products, or at the max, a few thousand, as opposed to the world of e-commerce where, there may be lakhs or millions of products. Let $n:$ Total No of distinct products

Let's define $D$ as the set of all the products we have, then,

$D=\left \{1, 2, 3, ..., n  \right \}$

![picture](https://drive.google.com/uc?export=view&id=1tpCqUbIkAwOsFs5t7QIpPVY3MnN4iwiW)

<br>

> **Q. Do we have a way of representing the items bought by a customer?**

Yes. Consider a customer that is done selecting the items they want, and are proceeding towards the billing counter. They present their **basket** full of products and the cashier scans the product bought and it's quantity.

This is called as a **transaction**, denoted by $T$.

For example,
- Suppose a customer bought item no. 1, 3, 6, and 8.
- This is represented as: $T1 = \left \{1, 3, 6, 8 \right \}$
- Similarly, there are other customers that bought some items as: <br>
$T2 = \left \{1, 3, 7, 12 \right \}$, <br>
$T3 = \left \{1, 7, 3, 16 \right \}$ <br>
... and so on till $m$ transactions (let).

If you think about it, $T$ is essentially a **subset of $D$**, i.e. $T_i ⊆ D$

**NOTE:**
- We are **not keeping track of the quantity** of a product bought by the user, in this transaction representation.
- Since everyday, a store like Walmart would see a footfall of 1000s of customers. In real world, this transaction would be very large.

<br>

> **Q. Could there be any pattern in what the customer is buying?**

Yes. There are some products that are more frequently bought together. For instance,
- Bread and milk
- Pen and notebook
- Toothbrush and paste
... and so on.

These are called as **item sets**.

As you can see from our transaction data above also, customers that are buying Item 1, also tend to buy Item 3. Here, the item set becomes: $\left \{1, 3 \right \}$

![picture](https://drive.google.com/uc?export=view&id=1HlFPKLF_kmKyL2kdRBnHv8eKX0nxDC9-)

<br>

> **Q. Why are these patterns important?**

Suppose there comes a customer that is buying Item1 but not buying Item3,
- since we already know based on a lot of transactional data, that these two are popularly bought together.
- We can recommend Item3 to the customer

![picture](https://drive.google.com/uc?export=view&id=1BN8ptIuM4wZYYExAf153jNXr4KXxlFAq)


#### Q. How to solve this problem?


*   **Transactional data is extremely large** and its very difficult to find patterns to identify which products are purchased together through a manual greedy approach

* Lets have a look at how transactional data looks like


In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from google.colab import files

In [ ]:
#importing CSV uploaded to drive
id = "1M5IPR96R9efi5cb2n8LxpXIXyx_F0x3v"
print("https://drive.google.com/uc?export=download&id=" + id)
!wget "https://drive.google.com/uc?export=download&id=1M5IPR96R9efi5cb2n8LxpXIXyx_F0x3v" -O Online_Retail.csv

https://drive.google.com/uc?export=download&id=1M5IPR96R9efi5cb2n8LxpXIXyx_F0x3v
--2026-01-19 06:55:42--  https://drive.google.com/uc?export=download&id=1M5IPR96R9efi5cb2n8LxpXIXyx_F0x3v
Resolving drive.google.com (drive.google.com)... 108.177.119.101, 108.177.119.102, 108.177.119.139, ...
Connecting to drive.google.com (drive.google.com)|108.177.119.101|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1M5IPR96R9efi5cb2n8LxpXIXyx_F0x3v&export=download [following]
--2026-01-19 06:55:42--  https://drive.usercontent.google.com/download?id=1M5IPR96R9efi5cb2n8LxpXIXyx_F0x3v&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 142.251.31.132, 2a00:1450:4013:c1a::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|142.251.31.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 45005763 (43M) [application/octet-stream]
Saving t

In [ ]:
df = pd.read_csv("Online_Retail.csv")
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,01/12/10 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,01/12/10 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,01/12/10 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,01/12/10 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,01/12/10 8:26,3.39,17850.0,United Kingdom


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


*   This datasets contains 8 features with 541909 rows!

#### Now let's formalise the task
**Given**
- $D$: Set of all items, and
- $T$: Set of all transactions

Note: No of transactions $m$ >> No of distinct items $n$

**To find:**
- **Item sets** that occur very frequently in transactions T.

![picture](https://drive.google.com/uc?export=view&id=1tRndSbYeha9weFX3jX7uvAavMghpggxD)


<br>

This technique of analyzing transaction data to give recommendations to the customer is called as **Market Basket Analysis**.

It is typically used in context of an offline store, where $n$ is not too large, it is typically a few hundreds.


## Data Preprocessing

Lets count the unique invoice numbers (transactions) and unqiue customer IDs

In [ ]:
print('Number of Unique Invoice numbers: {cnt}'.format(cnt=df.InvoiceNo.nunique()))
print('Number of Unique Customer IDs: {cnt}'.format(cnt=df.CustomerID.nunique()))

Number of Unique Invoice numbers: 25900
Number of Unique Customer IDs: 4372


In [ ]:
df.describe()

,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


> **Q. Why are some values negative under the `Quantity` and `UnitPrice` columns?**

These denote returned or cancelled items.

In order for us to find patterns and analyse the transactional data, we need not consider the items that were returned, we need only look at the items bought by a user in one go.

<br>

Hence, Let's get rid of all rows where quantity is `< 0`.

In [ ]:
df = df[df['Quantity']>=0]
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 531285 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    531285 non-null  object 
 1   StockCode    531285 non-null  object 
 2   Description  530693 non-null  object 
 3   Quantity     531285 non-null  int64  
 4   InvoiceDate  531285 non-null  object 
 5   UnitPrice    531285 non-null  float64
 6   CustomerID   397924 non-null  float64
 7   Country      531285 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 36.5+ MB


Let’s drop the rows that don’t have invoice numbers and remove the credit transactions (those with invoice numbers containing C).

In [ ]:
df.dropna(axis=0, subset=['InvoiceNo'],inplace=True)
df['InvoiceNo']=df['InvoiceNo'].astype('str')
df = df[~df['InvoiceNo'].str.contains('C')]
df.shape

(531285, 8)

Let's explore the column `Country`

In [ ]:
df['Country'].value_counts()

,count
Country,
United Kingdom,486286
Germany,9042
France,8408
EIRE,7894
Spain,2485
Netherlands,2363
Belgium,2031
Switzerland,1967
Portugal,1501


Since most our data is from UK, let's just consider that and drop the rest.

Also, let's convert this data to the form of a **sparse matrix** such that:
* We encode the basket data into a binary data that shows whether an items is bought (1) or not (0)


In [ ]:
data = (df[df['Country'] =="United Kingdom"].groupby(['InvoiceNo', 'Description'])['Quantity']
               .sum().unstack().reset_index().fillna(0).set_index('InvoiceNo'))
data.head(2)

Description,4 PURPLE FLOCK DINNER CANDLES,50'S CHRISTMAS GIFT BAG LARGE,DOLLY GIRL BEAKER,I LOVE LONDON MINI BACKPACK,NINE DRAWER OFFICE TIDY,OVAL WALL MIRROR DIAMANTE,RED SPOT GIFT BAG LARGE,SET 2 TEA TOWELS I LOVE LONDON,SPACEBOY BABY GIFT SET,TOADSTOOL BEDSIDE LIGHT,...,returned,taig adjust,test,to push order througha s stock was,website fixed,wrongly coded 20713,wrongly coded 23343,wrongly marked,wrongly marked 23343,wrongly sold (22719) barcode
InvoiceNo,,,,,,,,,,,,,,,,,,,,,
536365,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
536366,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


*   For any transaction, if the quantity of an item is >=1 then its encoded as 1 (bought) , else 0 (not bought)


In [ ]:
def encode_units(x):
    if x <= 0:
        return 0
    if x >= 1:
        return 1

data = data.applymap(encode_units)
data

Description,4 PURPLE FLOCK DINNER CANDLES,50'S CHRISTMAS GIFT BAG LARGE,DOLLY GIRL BEAKER,I LOVE LONDON MINI BACKPACK,NINE DRAWER OFFICE TIDY,OVAL WALL MIRROR DIAMANTE,RED SPOT GIFT BAG LARGE,SET 2 TEA TOWELS I LOVE LONDON,SPACEBOY BABY GIFT SET,TOADSTOOL BEDSIDE LIGHT,...,returned,taig adjust,test,to push order througha s stock was,website fixed,wrongly coded 20713,wrongly coded 23343,wrongly marked,wrongly marked 23343,wrongly sold (22719) barcode
InvoiceNo,,,,,,,,,,,,,,,,,,,,,
536365,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536366,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536367,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536368,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536369,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
581585,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
581586,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
A563185,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Since our goal is to find frequently occuring items, let's get rid of all transactions where only one product is bought.

We are going to uncover the association between 2 or more items that is bought according to historical data


In [ ]:
data = data[(data > 0).sum(axis=1) >= 2]
data

Description,4 PURPLE FLOCK DINNER CANDLES,50'S CHRISTMAS GIFT BAG LARGE,DOLLY GIRL BEAKER,I LOVE LONDON MINI BACKPACK,NINE DRAWER OFFICE TIDY,OVAL WALL MIRROR DIAMANTE,RED SPOT GIFT BAG LARGE,SET 2 TEA TOWELS I LOVE LONDON,SPACEBOY BABY GIFT SET,TOADSTOOL BEDSIDE LIGHT,...,returned,taig adjust,test,to push order througha s stock was,website fixed,wrongly coded 20713,wrongly coded 23343,wrongly marked,wrongly marked 23343,wrongly sold (22719) barcode
InvoiceNo,,,,,,,,,,,,,,,,,,,,,
536365,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536366,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536367,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536368,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536372,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
581582,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
581583,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
581584,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


According to the result above, we could see that there are 16,539 transaction that bought more than 1 items. It means, 91 % of the basket data is a transaction that has bought more than 1 item


## Apriori Algorithm

> **Q. How can we find the frequent item sets from Transaction data $T$?**

Let's create **key value pairs**, where:
- **key:** represents the **item-sets** / pairs of items bought together
- **value:** represents the **count** of occurence of the key item set

From this representation, we are using the transaction data to get a fair sense of items that are frequently bought together.

Item sets that have a very high frequency would have a relation, so based on that we can recommend items to the customer. For example:
- It was found that young adults between 5pm-7pm, tend to buy **beers** and **diapers** together.
- This is a very surprising relation, that **could not have been guessed!**
- Utilizing this fact, as a Data scientist at Walmart, we can recommend the stores to keep diapers section close to where beers are kept.

![picture](https://drive.google.com/uc?export=view&id=1YjTFmMKVepn18Gj2wXMlvWQjGBVAeb-e)

<br>

> **Q. How will we get the key values consisting of item sets?**

We are given the set of all products $D$. We can list out all of its subsets to get all possible item sets.

For example,
- If $D= \left \{1, 2, 3\right \}$, then all the subsets are: $\left \{1 \right \}, \left \{2\right \}, \left \{3 \right \}, \left \{1, 2 \right \}, \left \{1, 3 \right \}, \left \{2, 3\right \}, \left \{1, 2, 3 \right \}$

Using this logic, we get all the subsets of $D = \left \{1, 2, 3, ..., n\right \}$

<br>

> **Q. How many subsets would $D$ have?**

Recall what we learnt in Math in school, the number of subsets of a set with $n$ elements is: $2^n$.

This is because, for all the $n$ items, there are 2 choices: To be present in the subset (1) OR be absent from the subset (0).

Which means, $2*2*2*2*...*2..n \ times = 2^n$.

This even includes the empty subset.

<br>

> **Q. Is this an ideal approach (i.e. finding all subsets of D, and taking the count of their occurences, to find relations between items.)? What will bethe time complexity?**

- In our context, n is the number of products in Walmart, which is a high number like a few hundreds.

 - Even if we consider $n=100$, number of subsets of $D$, become $2^{100}$, which is an insanely high number.

- Besides that, we will have to scan through all the $m$ transaction sets,  
 - where recall that $m$ is the total number of transaction data and that it might be in millions.

This makes the time complexity as: $O(2^n *m)$

This is very tough to compute, it'll be very **very slow.**


![picture](https://drive.google.com/uc?export=view&id=1PR_XnhTpnmWdiNfOFD3GTzKAtH3GBYvA)

#### Q. How can we optimise on this idea?

To think of a possible optimization, first let's consider an example to get an idea across.


Using this idea, if we have an item set, we can use a **threshold value** (aka **minimum support**).

This makes sense, if we have millions of transaction data sets, we would like to set some sort of **minimum support**, beyond which, we can be confident about an item-set having a repeating pattern, as at least, that many number of customers are known to have bought those items together.

This minimum support is denoted as: $c$

![picture](https://drive.google.com/uc?export=view&id=1HhHhiQj3FJuRpWPqsOmCLpNn_q__FAO5)

<br>

> **Q. How can we use the intuition of minimum support to optimize our solution approach?**

- While listing down all the subsets of our items, $D$, say we come across a set $A$.
- Suppose the number of occurences of $A$ in the transaction data $T$ is less than the minimum support value, ie. $<c$
- Then, we need not consider $A$, or any of it's subsets.

For example:
- Let $c=100$, and we have the $D = \left \{1, 2, 3, 4\right \}$ and $T$ data.
- First, we list all the sets of size 1
 - $\left \{1\right \}, \left \{2\right \}, \left \{3\right \}, \left \{4\right \}$

- Now, we find the number of occurrences of each of these,
 - Let that be equal to $110, 200, 150, 50$ respectively.

- Since the number of occurrence of $\left \{4\right \}$ is less than $c$, no superset of it will occur more than 100 times, so we can ignore all of it's supersets.

- Now, we find subsets of size 2, for all sets which have occurred more than c
 - $\left \{1, 2\right \}, \left \{1, 3\right \}, \left \{2, 3\right \}$

- Again, we find number of occurences of each of these:
 - Let that be: $105, 110, 60$

- Since $60<c$, $\left \{2,3\right \}$ is **not** a frequent item-set

- Now, we build sets of size 3,
 - We know that $\left \{2, 3\right \}$ and $\left \{4\right \}$ cannot be a subset of it
 - So, the only element left is $1$, we cannot create set of length 3 with just that
 - So, there exist no superset of length 3.

![picture](https://drive.google.com/uc?export=view&id=1_x2euKN0ET9gABTIIdnHChKHUO61PQO4)

<br>

This way, though we'll still have to parse throught the $m$ transaction data for each item-set, we're able to significantly reduce the number of item-sets ($2^n$).

![picture](https://drive.google.com/uc?export=view&id=1i7CFNb7WPGhZtAZ_3fxTPgmnKJK7hVx_)


This approach is known as the **Apriori Algorithm**, and it was developed around 1994-95.

Though it is very simple in design, as soon as $n$ increases, it becomes too costly to be productive.

**The worst case complexity is still: $O(2^n*m)$**

Hence, it is not used in e-commerce today, where the number of products is very very large.

![picture](https://drive.google.com/uc?export=view&id=1lPWhsCefcn3aK6d-ryGuPMsxDyvswu0r)

<br>

However, modifications have been made over the concept of Apriori algorithm with time. This has given rise to a new technique:-
- **Frequent Pattern (FP) growth items**
 - Uses specialised data structure of **tries**
 - It is faster

Regardless of the modifications of FP growth, essentially we're still performing **frequency item-set mining**
- These are still very very expensive
- It is useful only when the number of items $n$ of $D$ is small
- Hence used in offline market places, where less products are present.
- Not scalable for world of e-commerce. We will learn other techniques for that soon.

![picture](https://drive.google.com/uc?export=view&id=1Or6_TJeeKtjHpyCE6sfYJ90Wve_kx_G-)


#### Q. Other than retail setups, can you think of other applications of Market Basket Analysis?

- **Bio-informatics**
 - If two chemical components $c_1$ and $c_3$ occur frequently within different proteins, then we can find out that perhaps there is some relation between these components
 - If two gene sequences $ATTC$ and $AGTC$ occur frequently, in the sequence of some mammal, then we can find out that perhaps there is some relation between them.

- **Medicine**
 - If we find that according to a doctor's presecription, medicines $m_1, m_2$ and $m_3$ are being prescribed frequently, then it means that together they form some combination drug, which can cure a certain ailment.

![picture](https://drive.google.com/uc?export=view&id=1K739RoCUtgyds9ZAEUv3ffgy73wJ93Xo)

- **Finding similar webpages / web usage mining**
 - If in a single session many users are visiting the same webpages ($w_1, w_2, w_3$), then perhaps they are related in nature

- **Finding similar words**
 -
 ![picture](https://drive.google.com/uc?export=view&id=1A9dclrQ8Xfr3mG7warRdE4bJS9Qtsuqj)


> **Q. How to implement Apriori Algorithm in code?**

We have a built-in function that implements apriori for us, under `mlxtend.frequent_patterns` library.

- As we discussed, we need to specify a minimum support threshold value, as parameter of this function.

- Since our column names represent the items, we use `use_colnames=True`

This will give us the most frequent item-sets, so let's sort them in desending order also, using `sort_values()`

Also, for easy interpretation, let's explicitly add a column stating the length of the itemset.

In [ ]:
from mlxtend.frequent_patterns import apriori

In [ ]:
frequent_itemsets_plus = apriori(data, min_support=0.03,
                                 use_colnames=True).sort_values('support', ascending=False).reset_index(drop=True)

frequent_itemsets_plus['length'] = frequent_itemsets_plus['itemsets'].apply(lambda x: len(x))

frequent_itemsets_plus

,support,itemsets,length
0,0.129875,(WHITE HANGING HEART T-LIGHT HOLDER),1
1,0.116331,(JUMBO BAG RED RETROSPOT),1
2,0.100671,(REGENCY CAKESTAND 3 TIER),1
3,0.095471,(PARTY BUNTING),1
4,0.084165,(LUNCH BAG RED RETROSPOT),1
...,...,...,...
172,0.030473,(CHRISTMAS CRAFT LITTLE FRIENDS),1
173,0.030473,(CREAM HEART CARD HOLDER),1
174,0.030413,"(LUNCH BAG BLACK SKULL., LUNCH BAG CARS BLUE)",2
175,0.030292,"(LUNCH BAG RED RETROSPOT, LUNCH BAG SPACEBOY D...",2


We get 177 most frequently occuring item-sets, using Apriori Algorithm!


*   Using apriori algorithm, we filter frequent itemsets by giving minimum support value of 3%
*   Length is the number of items in the itemset
* Based on the support threshold of 3%, there are 177 itemsets that are considered as frequently bought
* Eg: White hanging Heart T-Light Holder is the most frequently bought items with the support value of 0.129875. i.e the item is bought 2148 times out of the whole transaction


---

## Association Rule

Using Apriori algorithm, we get a sense of which item-sets are frequent.

We have another tool in the Market Basket Analysis, called the **Association rule**, using which we can find relations between these frequent item-sets.

<br>

> **Q. How does the Association rule work?**

Consider we have our set of items: $D = \left\{ 1, 2, 3, ..., n\right\}$

Let's define X and Y as another sets of items, as follows:-
- $X= \left \{1, 2, 3 \right\}$
- $Y = \left\{ 4, 6 \right\}$

If the item-set $\left \{1, 2, 3, 4, 6\right \}$ is a **frequent itemset**, then according to the **Association Rule** of Market Basket Analysis, we can say that
- "People who buy $X$, have a very high likelihood to buy $Y$ also.

This can be written as: $X -> Y$

**It is read as: "If X, then Y".**

![picture](https://drive.google.com/uc?export=view&id=1DzwEdj6dgmy3Vb4N7cMTHYJzaOmEaVqE)

<br>

> **Q. What are some real life examples of association rule?**

For example:-
- **If** a person buys beer, **then** there is a high tendecy of buying diapers
 - $\left\{ beer \right \} -> \left\{ diapers \right\}$

- People buying milk and bread, also tend to buy jam and eggs
 - $\left\{ milk, bread \right \} -> \left\{ jam, eggs \right\}$

![picture](https://drive.google.com/uc?export=view&id=1AOrVfsR0J022K6EnErXfYzpZv3o2lA0F)

<br>

> **Q. Can X and Y be used interchangeably?**

**No.**

$X → Y$ is not the same as $Y → X$

When we say, People buying beer, have high tendecy of buying diapers, that **does not imply** that people buying diapers, have a high tendency of buying beers.

In order to set these apart, we have the following terminologies in place:
- **Antecedent (If):** The items on the LEFT ie., the item which the customer buy
- **Consequent (Then):** The items on the RIGHT ie., the item which the customer follows to buy.

Consider that you are at Domino's,
- It is more likely for a customer to buy combination of **pizza + coke**
- than a combination of **pizza + garlic bread**

Though, both these associations rules hold true:-
- $\left\{ pizza → coke \right \}$
- $\left\{ pizza → garlic \ bread \right \}$

We know that one of them is more strongly associated than the other.

<br>

### Q. Are there any metrics to know how strongly two itemsets are associated?

Yes, there are a couple of different metrics that give us a better idea about associations. These are:-
- Support
- Confidence
- Lift
- Leverage
- Conviction

Lets take a look at them one by one.

---

#### Support
We talked about setting a minimum support threshold in Apriori Algorithm, let's formally introduce the concept of **support**.

> **Q. What is support?**

Support is a metric of How frequently does an item or item-set occur in the transaction data

<br>

> **Q. How is support calculated?**

This is calculated by dividing the number of transactions of where an itemset $X$ has occured in the transaction (let this be x), by the total number of transactions (let this be N)

$Support(x) = \frac{x}{N}$

<br>

> **Q. How can we interpret a support value?**

From a probabilistic standpoint, and Even if you see the formula,
- support is essentially the probability of an item-set $X$ occuring in the transaction data $T$.

![picture](https://drive.google.com/uc?export=view&id=1hyQZl6dureqy5t-g8O3kZhswBldsNw0p)

#### Confidence

Using Association rules, we stated that a person buying $X$ also tends to by $Y$, where $X$ and $Y$ are item sets.

> **Q. Since we need to base some business decisions on this, How confident are we in this statement?**

If we know that people buying milk and bread also tend to buy jam and eggs, then we can make a business decision and place all these items very close to each other.

But in order to make that decision, we first need to know how sure in this.

This can be answered by a term in market basket analysis called **confidence**

Confidence tells about the number of times these relationships have been found to be true

<br>

> **Q. How can we calculate confidence?**

This can be calculated by dividing the number of times both $X$ and $Y$ occur in transactions, by the number of times just $X$ occurs in transactions.

$confidence(X->Y) = \frac{Number \ of \ transactions \ with \ X \ and \ Y}{Number \ of \ transactions \ with \ X}$


<br>

> **Q. How can we interpret a confidence value?**

- This can be thought of as probability of $Y$ conditioned on $X$, $P(Y|X)$

 - i.e. Of all the times that $X$ occurs, how many times do we observe $Y$.

 - If $X U Y$ is a frequent itemset itself, then the confidence will be very high $≈90$%

- The confidence measure helps identify which product drives the sale of which other product.
 - For any two products, A drives B  {A ⇒ B} is not the same as B drives A, {B ⇒ A}


![picture](https://drive.google.com/uc?export=view&id=1scu8N-ELZR6KgcipA614kWdknpq9nG5w)

**Note:**
- A rule may show a strong correlation in a data set because it appears very often but may occur far less when applied (i.e checked against the antecedent).
 - This would be a **case of high support, but low confidence**.

- Conversely, a rule might not particularly stand out in a data set, but continued analysis shows that it occurs very frequently.
 - This would be a **case of high confidence and low support**.

#### Lift

Consider the combination: {Cornflakes} → {Milk}
- This should be a high confidence rule.

<br>

> **Q. What about {Yogurt} → {Milk}?**

High again.

<br>

> **Q. What about {Toothbrush} → {Milk}?**

Not so sure?
- Confidence for this rule will also be **high** since {Milk} is such a frequent itemset and would be present in every other transaction.

- It does not matter what you have in the antecedent for such a frequent consequent.
- The confidence for an association rule having a very frequent consequent will always be high

Analyse this:
<ul>
<li>Total transactions = 100
<li> 80 of them have milk
<li> 14 of them have toothbrush
<li> 10 of them have both milk and toothbrush
</ul>

Confidence for {Toothbrush} → {Milk} will be 10/14 = 0.7

Looks like a high confidence value. But we know intuitively that these two products have a weak association and there is something misleading about this high confidence value.

<br>

> **Q. How can we overcome this problem?**

Since, Considering just the value of confidence limits our capability to make any business inference.

**Lift** is introduced to overcome this challenge.





> **Q. What is lift?**

Let’s run another mini analytics.
Suppose an X store’s retail transactions database includes the following data:
<ul><li>
Total number of transactions: 600,000
<li>Transactions containing Bread: 7,500 (1.25 percent)
<li>Transactions containing Milk: 60,000 (10 percent)
<li>Transactions containing both Bread and Milk: 6,000 (1.0 percent)
</ul>

From the above figures, we can conclude that
- if there was no relation between Bread and Milk (that is, they were statistically independent),
 - then we would have got only 10% of Bread purchasers to buy Milk too.

- However, as surprising as it may seem, the figures tell us that 80% (=6000/7500) of the people who buy Bread also buy Milk.

This is a significant jump of 8x over what was the expected probability.

This **factor of increase is known as Lift** – which is the ratio of the observed frequency of co-occurrence of our items and the expected frequency.

Based on the low percentages we are seeing here (1.25%, 10%, 1%), we would have expected a low lift%.  

However, the fact that about 80% of Bread purchases include the purchase of Milk indicates a link between Bread and Milk.

TODO: Scribble 1



<br>

> **Q. How is lift calculated?**


It is based on the idea that, if $X$ and $Y$ are independent events, then the probability of $X and Y$ is equal to the product of their individual probabilities: $P(X ∩ Y) = P(X) . P(Y)$

Lift can be calculated by dividing the support of X and Y, by their individual support values.

$lift(X → Y) = \frac{support(X ∩ Y)}{support(X).support(Y)} = \frac{confidence(X → Y)}{support(Y)}$

This can be interpreted as being the same as: $\frac{P(X ∩ Y)}{P(X) . P(Y)}$

![picture](https://drive.google.com/uc?export=view&id=1pvRYIHM_ox9WpVEu0YCXFjLTzKqMOZNs)

<br>

> **Q. What happens if X and Y are actually independent?**

In that case, the numerator and denominator will be both equal, and give a result of 1.

Otherwise, the numerator will be greater.

- $lift(X→Y) = 1$, if X and Y are independent
- $lift(X→Y) < 1$, unlikely to be bought together: **negative correlation**
- $lift(X → Y) > 1$, likely to be bought together: **positive correlation**

**Note:**
- The value of lift ranges from **0 to infinity**.

![picture](https://drive.google.com/uc?export=view&id=1664Io4QPmIZYy6R-3vsa4eZk1ki5IaF1)

Now if we analyse the two itemset pair where both had high confidence :

$Lift\left\{Bread -> Milk\right\} = \frac{Confidence \left\{Bread -> Milk\right\}}{ Support(Milk)}$
$ = \frac{6000/7500}{60,000/600,000} = 80$

<br>

Similarly, we get

$ Lift \left\{Toothbrush → Milk \right\} = \frac{Confidence \left\{Toothpaste -> Milk\right\}}{ Support(Milk)} = \frac{0.7}{0.8} = 0.87$

<br>
A value of lift less than 1 shows that having toothbrush on the cart does not increase the chances of occurrence of milk on the cart in spite of the rule showing a high confidence value




#### Leverage

There is another metric called **leverage** that is constructed using support. This is very similar to lift.

<br>

> **Q. How is leverage calculated?**

To compute the leverage of "if X then Y" i.e. $X->Y$, we compute the support of $X and Y$, and subtract the product of support of X, and support of Y from it.

$leverage(X->Y) = support(X \cap Y) - (support(X)*support(Y))$

<br>

> **Q. What is the advantage of using leverage over lift values**

- Though it is similar to lift, but leverage is **easier to interpret**.
- Leverage value lies in the range of **-1 to +1**, whereas lift value ranges from 0 to infinity.

#### Conviction

> **Q. How can we calculate conviction value?**

It can be calculated as the ratio of the expected frequency that X occurs without Y if X and Y were independent divided by the observed frequency of incorrect predictions.

$Conv (X → Y) = \frac{1 - S(Y)}{1 - C(X → Y)}$

<br>

> **Note**

A high value means that the consequent depends strongly on the antecedent.


Now that we've looked at assocation rules, and it's metrics, let's implement it in code.

<br>

> **Q. How to implement this in code?**


- After applying the apriori algorithm and finding the frequently bought item, we apply the association rules.
- From association rules, we could extract information about which items are more effective to be sold together

We have a built-in function for this as well in the same library: `mlxtend.frequent_patterns`.

**Note:**
- We pass the frequent item sets we got from apriori algo as the parameter here.

In [ ]:
from mlxtend.frequent_patterns import association_rules

In [ ]:
association_rules=association_rules(frequent_itemsets_plus, metric='lift',
                  min_threshold=1).sort_values('lift', ascending=False).reset_index(drop=True)
association_rules.head(5)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction
0,(GREEN REGENCY TEACUP AND SAUCER),(PINK REGENCY TEACUP AND SAUCER),0.056473,0.042264,0.034887,0.617773,14.617093,0.032500,2.505674
1,(PINK REGENCY TEACUP AND SAUCER),(GREEN REGENCY TEACUP AND SAUCER),0.042264,0.056473,0.034887,0.825465,14.617093,0.032500,5.405948
2,(PINK REGENCY TEACUP AND SAUCER),(ROSES REGENCY TEACUP AND SAUCER ),0.042264,0.057682,0.033013,0.781116,13.541798,0.030575,4.305101
3,(ROSES REGENCY TEACUP AND SAUCER ),(PINK REGENCY TEACUP AND SAUCER),0.057682,0.042264,0.033013,0.572327,13.541798,0.030575,2.239413
4,(GARDENERS KNEELING PAD CUP OF TEA ),(GARDENERS KNEELING PAD KEEP CALM ),0.045287,0.054235,0.032711,0.722296,13.317793,0.030254,3.405662


## Q. How can we draw Data Conclusions from these associations rules?


*   We can see GREEN REGENCY TEACUP AND SAUCER and PINK REGENCY TEACUP AND SAUCER are the items that has the highest association each other since these two items has the **highest lift value**, i.e 14.6

 *   This tells us that 'PINK REGENCY TEACUP AND SAUCER' is 14.6 times more likely to be bought by the customers who buy 'GREEN REGENCY TEACUP AND SAUCER' compared to the **default likelihood sale** of 'PINK REGENCY TEACUP AND SAUCER’

* Since the antecedent support > consequent support, the rule that applies is **{GREEN REGENCY TEACUP AND SAUCER -> PINK REGENCY TEACUP AND SAUCER}**

 * This means that  a customer has a higher tendency to buy PINK REGENCY TEACUP AND SAUCER AFTER they buy GREEN REGENCY TEACUP AND SAUCER. Not  the other way around

* The confidence level for the rule is 0.6177, which shows that out of all the transactions that contain both “GREEN REGENCY TEACUP AND SAUCER”, ~62 % contain PINK REGENCY TEACUP AND SAUCER too

 * This could  very valuable information, because we are now aware which products should we put the discounts on. We could give discounts on PINK REGENCY TEACUP AND SAUCER if a customer buy GREEN REGENCY TEACUP AND SAUCER etc

Similarly, we can draw other business conclusions from these association rules, which will lead in increase in revenue!


## Q. What Business Strategy can we follow based on these findings?



*   **Item Placements:** We could place these two products together for easy accessiblity and increase sales

*   **Product Bundling:** We could bundle these two products and sell it together at a discounted price compared to each price combined

* **Customer Recommendation & Discounts:** We could place PINK REGENCY TEACUP AND SAUCER at the cashier, and every time a customer buys GREEN REGENCY TEACUP AND SAUCER, we could offer and recommend them to buy PINK REGENCY TEACUP AND SAUCER with a lower price



## Summary

- Market basket analysis is a business problem, which can be solved using **Apriori algorithm / FP growth algorithm**.
- Also, market basket analysis can be interpreted in the context of **association rule mining**
- To find these association rule mining, and frequent item sets, we will still go to appriori and FP growth algos

![picture](https://drive.google.com/uc?export=view&id=1b1lWMUFDgsID4fj07ifFmjhudf6IGszE)

## Recommendation System Introduction

#### Q. What is a recommendation system/engine?
A recommender system is a system which predicts ratings a user might give to a specific item.

These predictions will then be ranked and returned back to the user as rcommendations.

The goal of a good recommmender engine is to hook the user to the platform.

There are many apps that we use that make use of recommendation systems. Example:
- Netflix
- Tiktok
- Amazon
- Youtube

<br>

> **Q. How have recommender systems evolved over time?**

- **Pre 2007**
 - **Similarity based**
 - **Content based**
 - **Collaborative Filtering** algorithms were used

- **2007 - 2015**
 - In 2007, **Netflix** held a competition where it promised a prize of million dollars to the team of people that can improve their recommendation engine.
 - The winning team had their solution based on the concept of **Matrix Factorization**, which became popular in this period.

- **Post 2015**
 - **Deep learning** algorithms are being used now.

![image](https://docs.google.com/uc?id=16kDmFNdeSUPoS4HyaC_AamF1ke7i7DvG)


#### Q. How do we formulate a recommender system problem?
The problem formulation here is different than what we've seen already for Classification or regression.

Suppose we have $n$ users, $U_i; i=[1, n]$ and $m$ items (product on amazon, movie on Netflix, song on spotify, etc), $I_j; j=[1,m]$.

`Goal:`
- We need to suggest a list of items to user $U_i$ that he/she will like.
- These items should be ranked (preferably).
- This is of course, based on historical data.

**NOTE:** Naturally, m would be in billions, whereas we can have millions of users (n). The point is, that the **scale is very large**.

![picture](https://drive.google.com/uc?export=view&id=1aSnd1pGoS6dOHVMi3p_py-9SPJ7QU976)

---

#### Q. How can we represent the dataset for recommender systems?

The dataset is represented in the form of a **matrix**, lets call it `A`.

In `A`, each row represents a user $U_i$, and each column represents an item $I_j$. This way, element $A_{ij}$ denotes user $U_i$'s interaction with product $I_j$.

This interaction can be measured in many ways, depending on context,
- If the youtube video is liked **(Binary value)**.
- Percentage of song listened to, in spotify **(real valued)**.
- If user spent 2 minuets reading about the product on amazon.
- Rating of the Netflix movie **(numeric value)**.
- If the product was bought or not.

Basically, we try to represent whether user indicated any form of interest in that product.

Naturally, `A` is a `n x m` matrix.

<br>

> **Q. What if user $U_i$ has never interacted with a specific product ,say $I_2$?**

When we have billions of products, it is next to impossible for any given user to have interacted with all of them.

For example,
- A user of Youtube, residing in New Delhi would not have seen any German videos, owing to the language barier.
- This leaves the entire section of German videos, that this user has never even interacted with.

In such cases, we leave the cell $A_{i2}$, where $I_2$ represents such a German videos, as **empty**.

Similarly, a lot of cells would be empty in matrix A.

![picture](https://drive.google.com/uc?export=view&id=1HQQpohiU06-jQB7zWZz8FGNrcAmOqfU8)



Recall that we have about a billion users, $n \approx 10^9$, and a few hundred million videos of Youtube, which means, $m ≈ 10^8$.

This means we have a total of $10^9 * 10^8 = 10^{17}$ cells in matrix A.

It is not possible for every user to have watched (or rated) every video on YouTube.

Practically, on an average, an user $U_i$ would have interacted with 1000 youtube videos, which is next to nothing in comparison to the total number of videos present.

Hence we can say that matrix A is **sparse**.

<br>

> **Q. How do we fix our matrix being sparse?**

We don't need to "fix" anything.

The main goal of building a recommender system is that if a user $U_i$ has given rating, for a few items, then the system should recommend new items based on the user's interests. This IS the problem at hand.

<br>

> **Q. How can we calculate the sparsity of matrix A?**

$sparsity = \frac{Number \ of \ non \ empty \ cells}{Total \ No. \ of \ cells}$

As per our example of Youtube,

$sparsity = \frac{10^9 * 1000}{10^{17}} = 10^{-5}$

This means that only 1 in 10<sup>5</sup> cells is not empty in A.

![picture](https://drive.google.com/uc?export=view&id=1cn2qXfypJveCvM64vJmyLssMwanT93tL)



> **ASSESSMENTS COVERED:-**
- https://www.scaler.com/hire/test/problem/26315/

---

## Collaborative Filtering

#### Q. We're just given the A<sub>n x m</sub> matrix, how can we recommend items to users based on this?
**Task:** Given A<sub>n x m</sub> , we need to recommend some items to user $U_i$.

Before we dive into this problem, first lets try to retrieve information about the users and the items from the given matrix A.


> **Q. How can we obtain an user / item vector from matrix A?**

We're just given the matrix A<sub>n x m</sub>, and we wish to obtain some representation of data for a user $U_i$.

Recall that for text data, we form a bag of words representation (BoW), which works as our features.

Similarly, we can have a vector of dimension $m$ x $1$, where each element would tell if user $U_i$ has bought/watched (shown interest in) item j, such that $j ϵ [1, m]$.

For cases where user $U_i$ has not interacted with / shown interest in item $I_j$, we can simply put 0.

This is a very crude approach, but this way we can have a user data.

![picture](https://drive.google.com/uc?export=view&id=1NbAvRsl8In3WuJhk_6TEeXfVr_6GKLcI)


<br>

If you think about it, this is essentially the ith row of matrix A.

Similarly, we can also get a $n$ x $1$ vector that represents data for an item $I_j$, which would be the jth column of matrix A.

![picture](https://drive.google.com/uc?export=view&id=1r4Gk3pqEFfWxbg40nFlW9Z01S8GVc1v7)

This way of using the given matrix A to form user and item vectors is known as **Collaborative filtering (CF)** reccomendation system.

Lets take a look at different approaches we can adopt to implement CF Recommendation systems.


---

## Item-Item based Similarity

We already have historical data about the items bought (positively rated) by a user $U_i$.

Let's say that this user has already bought item $I_{10}$ and $I_{12}$

> **Q. What if we find an item $I_j$ that is similar to these already bought items?**

This is a good idea.

We can find items with high similarity to the items historically bought by user $U_i$ ($I_{10}$ and $I_{12}$), as we know that this user showed interest in such items.


For example,
- If a user of Spotify likes music by Lata Mangeshkar, there is a high chance he/she will like songs by Asha Bhosle also.
- If a user of Netflix likes Mission Impossible 2, there is a high chance he/she will like other action movies like John Wick.

There is a sense of similarity among these items.

In our case,
- since our user $U_i$ likes $I_{10}$ and $I_{12}$,
- we need to find set of items similar to both $I_{10}$ and $I_{12}$
- If an item $I_k$ lies in both these sets, there is more chances of user $U_i$ liking item $I_k$.

This is like the **Nearest Neighbour** based approach.

<br>

> **Q. How can we find similarity between different items?**

Every $I_j$ is a n dimensional vector. It represents the ratings given to item $I_j$ by all the n users.

So, we use **cosine similarity** metric to measure the similarity between a specific item say $I_{10}$ and rest of the items.

Based on this metric value, we recommend the items that are most similar to $I_{10}$.

$sim(I_i, I_j) = cosine \ similarity(I_i, I_j) = \frac{I_i ^T * I_j}{||I_i|| * ||I_j||} = S_{ij}$


![picture](https://drive.google.com/uc?export=view&id=1GS4UyFjq2dN-YullCGZd9P1euM48eT0T)


<br>

> **Q. How can we store these similarity scores?**

We build a **similarity matrix**, with all the $S_{ij}$ values.

Since we're talking about item-item similarity, we denote this matrix as $S^i$.

The dimensions of $S^i$ becomes $m$ x $m$, and $S_{ij}^i$ represents how similar two items $I_i$ and $I_j$ are.

Larger the value of $S_{ij}^i$, more similar items $I_i$ and $I_j$ are.

TODO: Scribble showing similarity matrix

<br>

This is known as **Item-Item based Collaborative Filtering**  Recommender system. It was introduced by **Amazon** in 1998.

Here, the recommender engine compares the items that are already positively rated by a user, with the items that he did not rate and looks for similarities.

Items similar to the positively rated ones will be recommended.

**NOTE:**
- This is one of the most basic ideas behind YouTube recommendation engine.
- There is a lot of added layers of complexity on top of this, but this is the idea.

<br>

The dimensions of the item-item similarity matrix is $m$ x $m$, and m is a very large number.

> **Q. Wouldn't the calculation of the similarity matrix be a very time consuming process?**

Yes. This would be very time consuming.

Back in the day, when this approach was used in platforms like Amazon, We could not afford to do spend so much time on the go, as the user logs in.

So, this calculation did not took place when the user logs in, rather it was performed during the nights, or maybe even weekly, so that when user logs in, they readily get the recommendations based on this calculation.

![picture](https://drive.google.com/uc?export=view&id=1hZWWGQHJL08dIoWWox8TngqexalbFquu)

---



## User-User based similarity

Consider a user $U_i$.

> **Q. What if we find a user $U_j$ that is similar to $U_i$?**

Since we know both these users are similar, we can use the history of one user to give recommendations to the other.

We already have a $m$ x $1$ vector representing user data for ith user, that we retrieved from A, $U_i$.

Using **cosine similarity** we can find similarity between user $U_i$ and all the others, to find the most similar users.

Here also, this data is formed in the form of a **similarity matrix** $S^u$, where $S_{ij}^u$ represents how similar user $U_i$ is to user $U_j$.

![picture](https://drive.google.com/uc?export=view&id=1gIo0rnPPZgYHmJDF6Rpgw54jKWh_5r5_)

<br>

> **Q. Using cosine similarity, we can find the most similar users, what now? How do we use this to recommend items to user $U_i$?**

Consider that user $U_i$ has already bought items $I_{10}$ and $I_{18}$.

Using cosine similarity, we find that users $U_{10}, U_{26}, U_{58}$ are similar to user $U_i$. Their purchase history includes:-
- $U_{10}$: $I_{10}$, $I_{12}$, $I_{18}$, $I_{20}$
- $U_{26}$: $I_{10}$, $I_{18}$, $I_{26}$, $I_{12}$
- $U_{58}$: $I_{18}$, $I_{12}$

User $U_i$ has already bought items $I_{10}$ and $I_{18}$.

Using a **frequency based** approach, we can say that item $I_{12}$ seems to be very popular among the other similar users ($U_{10}, U_{26}, U_{58}$), we can guess that user $U_i$ might also be interested in buying $I_{12}$, making it a good recommendation!

![picture](https://drive.google.com/uc?export=view&id=10sy_RR9__4hswrL8uyG-NNJR42Ehej8-)

<br>

This technique, where we find similar users is called **User-User similarity based Collaborative Filtering**.

**NOTE:**
- This approach is also similar to Nearest Neighbour approach.

<br>

> **Q. Can you think of a flaw in user-user similarity approach?**

One major problem with user-user similarity is that **User preferences can change over time**.

This can lead to bad recommendations.

In order to handle this, we prefer using item-item similarity approach, because in contrast, the ratings on given items do not change significantly over time.

> **Q. How to decide when to use user-user or item-item similarity approach?**

Consider the following rule of thumb.

When we have more number of users than items, i.e. n > m, and if the item ratings do not change much over time, after the initial period, then it is better to use the item-item similarity approach.

<br>

---

## Cold Start Problem

> **Q. What if we have a new user joining the platform?**

When a new user joins, then we have no ratings given from this user, as he is new to the ecosystem. So, we cannot find similar users.

Suppose $U_i$ is a new user. Hence, the cells of ith row in A would be empty.

This means that entire user vector has no data.

For every recommender system, it is required to build a user profile by considering the user's activities and behaviours with the system.

Based on user's history only, recommendations are made.

If there is no historical data, that is a problem.

<br>

> **Q. Similarly, what if there is a new product that's added to the platform?**

Here again, we have no historical data, as it is a new product.

We cannot find any similar items to it.

If $I_j$ represents the new item, jth column of A would be empty.

This means that entire item vector has no data.

![picture](https://drive.google.com/uc?export=view&id=1vR2xZ3AgfRWZq1hyBl38nlKkFQ0fT4U1)


<br>

> **Q. What is a cold start problem?**

Both these cases are called as a **Cold Start Problem**.

This arises since we have no data about these new users/items, hence preventing us from being able to give good recommendations.

This problem arises due to 3 different reasons:-
- For new users
- For new items
- For new communities

In these cases, we don't have enough information to make good decisions/recommendations.

<br>


> **ASSESSMENTS COVERED: (COLLABORATIVE FILTERING)**
- https://www.scaler.com/hire/test/problem/17187/
- https://www.scaler.com/hire/test/problem/17188/
- https://www.scaler.com/hire/test/problem/20338/
- https://www.scaler.com/hire/test/problem/19931/

> **ASSESSMENTS COVERED: (COLD START)**
- https://www.scaler.com/hire/test/problem/17092/
- https://www.scaler.com/hire/test/problem/26314/
- https://www.scaler.com/hire/test/problem/26329/

> **ASSESSMENTS COVERED (SIMILARITY BASED):-**
- https://www.scaler.com/hire/test/problem/26317/
- https://www.scaler.com/hire/test/problem/26319/
 >> **Coding Questions**
- https://www.scaler.com/hire/test/problem/26460/

---

## Content based Recommendation System


> **Q. How can we overcome cold start problem?**

A basic idea would be to recommend the most popular / frequently bought items using a frequency based approach. But this is a very vague approach. Let's think of something else.

Consider the case of a new user.

Even though we do not have any information regarding this user's interactions with different items, we do have other additional information about this new user.
- **Location**
 - This can be used to get an idea of the items used / purchased by other users in that area.
 - A swiggy recommender engine can make an assumption that Idli-Dosa are more probable to be liked by a user residing in Southern India.
 - We can get the location of user from IP Address
 - Most platforms do ask for your location before letting you sign up.

- **Gender**
 - Useful in recommending clothes and accessories.

- **Age**

- **Type of Credit Card**
 - This too can help get a lot of information about the data, like their spending habbits, their credit limit, brand of credit card, etc.

- **Device being used to access the platform**
 - We can assume that an user using Apple Macbook would have more spending power than a user using a cheap Chinese smartphone.

We form a new d'-dimensional vector that holds all this data, and then use **user-user similarity** on it, and recommend accordingly.

This is known as **User-user similarity based Content Filtering** Recommender systems.


![picture](https://drive.google.com/uc?export=view&id=1_h3afZD7o4PYRpiO3SJ5anALn-Z_1-8H)

<br>

> **Q. Do we have any kind of additional information that can be used in case of a new item cold start?**

Yes.

Consider that there is a new product on Amazon, though there is no data about user ratings, we still have additional information like:-
- **Product description**
 - This would potentially be stored as a BoW
- **Price**
- **Category of product**
 - Like electronics, clothing, sports, etc.
... and so on.

We form a new d-dimensional vector that holds all this data, and then use **item-item similarity** on it, and recommend accordingly.

Hence this is called **item-item similarity based content filtering** Recommender systems.

We can recommend this new item to those users who bought similar items from the same categories until we have sufficient information.

![picture](https://drive.google.com/uc?export=view&id=1BdcUIWKgog9doqXhOXWc8F-JgSPqJrs-)

<br>

These additional information is known as **metadata**.

This process of finding user-user or item-item similarities, using metadata in order to recommend items to users is called **Content based Recommendation system**.

<br>

> **Q. Why is it called "Content based" Recommendation?**

Because we are not using the purchase data (the $A_{ij}$s) for finding similarity and then recommending.

Instead we are using **features extracted from**/provided in the **content** of the user / item to form d-dimensional vector representing the user/item metadata.

The point of content based filtering is that we have to know the content of both the users and the items.


![picture](https://drive.google.com/uc?export=view&id=1jLtzH5t0VkCJLYVPtOFXznvEE2lfrP2a)



#### Q. What are some advantages and disadvantages of content based filtering?

**Advantages**
- The model doesn't need any data about other users, since the recommendations are specific to this user. This makes it **easier to scale** to a large number of users.
- The model can capture the specific interests of a user, and **can recommend niche items** that very few other users are interested in.

**Disadvantages**
- Since the feature representation of the items are hand-engineered to some extent, this technique requires a lot of domain knowledge. Therefore, the model can only be as good as the hand-engineered features.
- It always recommends items related to the same categories, and never recommend anything from other categories.

<br>

#### Q. What are some advantages and disadvantages of collaborative filtering?
**Advantages**
- No domain knowledge necessary
 - We don't need domain knowledge because the embeddings are automatically learned.
- **Serendipity**
 - The model can help users discover new interests. In isolation, the ML system may not know the user is interested in a given item, but the model might still recommend it because similar users are interested in that item.

- The system doesn't need contextual features.

**Disadvantages**
- Fails in case of cold start.

> **ASSESSMENTS COVERED:**
- https://www.scaler.com/hire/test/problem/17052/
- https://www.scaler.com/hire/test/problem/17051/
- https://www.scaler.com/hire/test/problem/17182/
- https://www.scaler.com/hire/test/problem/20187/
- https://www.scaler.com/hire/test/problem/19930/

---

## Recommendation as a Regression/Classification problem

We saw how we can use the metadata to create d-dimensional user and item feature vectors.

> **Q. Can we use these d-dimensional user and item vectors as features, of a Regression/Classification problem?**

Suppose that a user $U_i$ gave a rating of 4 to an item $I_j$, i.e. $A_{ij} = 4$.

Using metadata, we can obtain a feature vectors for the user $U_i$ and for item $I_j$ of dimension d' and d respectively.

If we concatenate these two feature vectors, and put the label as $A_{ij}$, which is the rating given by $U_i$ to $I_j$, we get an entry of data that we can use for training a regression/classification problem.

This way, we can form training data from the entries of matrix A, that are non-empty.

Here, we're using both, the metadata given to us, and the entries of matrix A, to train a regression/classification problem, and predict the ratings of a new pair of user and item, given their features.

Therefore, this becomes a hybrid model of sorts.

![picture](https://drive.google.com/uc?export=view&id=1nQaCej8Y31ZBWH0eT1UTY0hPHBPhY22h)

<br>

> **Q. Is there any problem with this approach?**

Consider the case when a new user is added.

We do not run into a cold start problem, as we can predict ratings of this new user for items, using his/her metadata.

But, in order to recommend the top 10 items, we will first have to predict the ratings for **ALL** the items. This is a problem because we can have as many as a million items in real world.

So, naturally, doing so would be very very computationally heavy.

Hence, this model is not feasible.

![picture](https://drive.google.com/uc?export=view&id=11ZLH5nKnyLK92gaa38jMkyLJNZdLT9EQ)


---

## Summary: Types of Recommendation Algorithm

The classical Recommender systems can be broken down into following types:-
1. Content based
 - Uses content based features like location, gender, etc
 - Very helpful in cold start problems
2. Collaborative filtering
 - Uses the data given in matrix A, i.e. the $A_{ij}$ values
 - Unlike content based filtering, we don't need to hand-engineer the features.
3. Hybrid models
 - Uses both content based features and  the $A_{ij}$ values

**Note:**
- Both user-user and item-item based similarity approaches can be adopted for both, content based and collaborative filtering techniques.

![picture](https://drive.google.com/uc?export=view&id=1o0dHtVRrI6mAcw2mn9tw7MTLIxYONCoK)


---

## Matrix Factorization (MF)

Recall the concept of factorization in algebra, that you'd have studied in school.

As per this concept, we can write a number as products of it's **factors**. For example, $6 = 2 * 3$.

Recall that we have our $n$ x $m$ matrix A.

Let's try to expand this concept for matrices by decomposing it as a product of two other matrices:-

A<sub>n x m</sub> = B<sub>n x d</sub> . C<sub>d x m</sub>

This is called as **Matrix Factorization (MF) / Matrix Decomposition**

<br>

> **Q. Can we factorize matrix A into a product of 3 matrices?**

Yes.

Just as 12 can be written as both $12 = 2*6$ and $12=2*3*2$, we can similarly decompose matrix A into 3 factor matrices instead of 2.

The final dimensions should match with $n$ x $m$. If that is taken care of, then we can factorize A into even more factors.

A<sub>n x m</sub> = B<sub>n x d</sub> . C<sub>d x d'</sub> . D<sub>d' x m</sub>

But, In context of Recommender systems, decomposition into 2 factor matrices is done.

![picture](https://drive.google.com/uc?export=view&id=1ZZ7AAx4WrK4gcKm2lvnXLM4wEt7rLs6p)

<br>

> **Q. How is the concept of MF relevant to Recommender systems?**

Recall that our matrix A is very sparse, which means that there are a lot of missing values.

If we could approximately complete this matrix based on the values that we do have, we would essentially get an idea of how each user would rate each item. This would be very helpful in recommending new items to the users.

This technique of utilising the available values to complete the sparse matrix A is called **Matrix Completion**.

There are tonnes of ways to solve the problem of Matrix Completion, one way of achieving this is by the process of **MF**.



![picture](https://drive.google.com/uc?export=view&id=13Oe-MaTwTBY4Yx4r_NvHNmgheQ5Orb5c)

![picture](https://drive.google.com/uc?export=view&id=1EV7IHts9dj-1AtLsGWDMNReo5wPF3Kaf)

> **Q. What is the underlying assumption behind Matrix Factorization in Recommender systems?**

The fundamental assumption of Matrix Factorization based Recommender systems is that $A_{ij}$, i.e. the user $U_i$'s rating for an item $I_j$ can be decomposed as a dot product between an user vector $B_i$ and an item vector $C_j$.

This idea of using Matrix Factorization for Recommender systems was introduced around 2008-2009, during the **Netflix prize competition**.

![picture](https://drive.google.com/uc?export=view&id=1CDcassowMBPLGITbNH0v0Om6470rcaSY)




> **Q. How can we go about completing the given matrix A using MF?**

Consider the matrix A. Even though we don't all of it's values as it is sparse.

Let's assume the following decomposition to be true: A<sub>n x m</sub> = B<sub>n x d</sub> . C<sup>T</sup><sub>d x m</sub>,
where <br>
n -> No of users <br>
m -> No of items


C has dimensions: $m$ x $d$, but in order to take a dot product, we need to take transpose, so that dimensions match with those of B<sub>n x d</sub>

This is known as an **Interaction model**, we will see why it's called that in a bit.

![picture](https://drive.google.com/uc?export=view&id=1ReFOAT3Zc7STUPyNqi1mdSvZjlhEqxjp)


<br>

#### Q. How can we represent the value of a cell $A_{ij}$ in terms of matrices B and C?

Look at how B and C matrix look like in the figure above.

We can obtain $A_{ij}$ by doing a dot product of the ith row of matrix B and the jth row of matrix C.

Let's obtain these rows in form of vectors. Recall that Whenever we write a vector, we say its a column vector.

So, we get $B_i$ and $C_j$ as the required column vectors respectively.

Hence, in order to do a dot product, we need to a transpose of $B_i$.


$A_{ij_{1 x 1}} = B_{i_{1 x d}}^T . C_{j_{d x 1}}$


<br>

Notice that our original matrix B had a dimension of $n$ x $d$, where n represents the no of users.

So we can interpret $B_i$ to be a d-dimenisonal vector that represents user $U_i$.

Similarly, $C_j$ can be thought of as a d-dimensional vector that represents the item $I_j$

TODO: Scribble 2
![picture](https://drive.google.com/uc?export=view&id=1yiNaCGWOfyrf0UhlA3yVIycBlfOECmWw)

**NOTE:**
- Multiplication in linear algebra is also known as "Interaction", hence we call this model as the **Interaction Model.**


As you can see we're getting $A_{ij}$, i.e. the user $U_i$'s rating for an item $I_j$ by the interaction (dot product, to be exact) between $B_i$ and $C_j$ vectors, that represent the user $U_i$ and item $I_j$ repectively.

This is the assumption we had made in the beginning of MF.

$A_{ij_{1 x 1}} = B_{i_{1 x d}}^T . C_{j_{d x 1}}$



#### Q. How can we find the $B_i$ and $C_j$ vectors, even though we only have a few $A_{ij}$ values in the rating matrix?

Suppose that $n=10,000$ and $m=1,000$

This means total number of cells in A = $10^4 * 10^3 = 10^7$

Since A is sparse, lets assume that out of these $10^7$ cells, only $10^5$ are non-empty.

**Task:** Given a small subset of $A_{ij}$ values, we want to compute d-dimensional vectors $B_i$ for all users ($i -> [1, n]$), and $C_j$ for all items ($j -> [1,m]$).

![picture](https://drive.google.com/uc?export=view&id=13j1uVjf7_82DJ8V_kOIQBOzlePhJnPrQ)


Though we don't know what $B_i$ and $C_j$ are, we want their product to be as close as possible to $A_{ij}$, for all the **non-empty** entries in A.

$A_{ij} ≈ B_{i_{1 x d}}^T . C_{j_{d x 1}}$

We can translate this in the form of **Mean Squared Loss**. We'd like to minimise that for all the cells that are **non-empty** in A

$min_{B_i, C_j} Σ (A_{ij} - B_i^T . C_j)^2$

This becomes our **optimization problem** for MF based Rec Sys.

![picture](https://drive.google.com/uc?export=view&id=1_hgbqyK2jrBoMvWctc5-kym3-A0XU7sn)

<br>

> **Q. How can we solve the optimization problem of MF based Rec Sys?**

There are broadly 2 approaches to solve this:-
1. **Stochastic Gradient Descent (SGD)**
 - Randomnly assign values in $B_i$ and $C_j$ and perform SGD
 - This will take time as there are huge number of users and items.
2. **Coordinate Descent Algorithms**
 - We consider $B_i$ and $C_j$ as separate coordinates
 - We fix $B_i$ for a few iterations and update $C_j$.
 - Then, we fix $C_j$ and update $B_i$ for a few iterations.
 - This is repeated untill they converge to solve the optimization problem.
 - This techinque is also known as **Alternating Least Squares (ALS)**

![picture](https://drive.google.com/uc?export=view&id=14fJvxyl_ntBtxSX9htJed3ql6Tq7mwL_)

So, using the non-empty cells of matrix A along with the optimization technique, we're able to find suitable values for $B_i$ and $C_j$.

Recall that this was the primary goal of this entire exercise.


<br>

> **Q. How can we complete the missing cells in matrix A?**

Suppose that the third user has not rated the tenth item, i.e. $A_{3, 10}$ -> Missing

Recall that our underlying assumption for MF based Rec Sys was that $A_{ij} \approx B_i^T . C_j$

We already computed the value for user vector $B_3$, using other ratings that user $U_3$ has given, which can be found in $A_{3, j}$.

Similarly, we have value for item vector $C_{10}$, as item $I_{10}$ would've been rated by other users, which can be found in $A_{i, 10}$.

So, taking a dot product of $B_3$ and $C_{10}$, can get a very close approximate value of the empty cell $A_{3, 10}$.

Similarly, we can fill up all the empty cells in matrix A.

> This is the relation between **matrix completion** and **recommender systems**.

![picture](https://drive.google.com/uc?export=view&id=1tPT40WRqwiE_AZl-MxlJqDiUdWSS0drZ)

Lets understand this with context to the optimisation problem, i.e. $min_{B_i, C_j} Σ (A_{ij} - B_i^T . C_j)^2$

**Suppose we're trying to find the item vector $C_{10}$**

In order to find this, we want to plug-in **ALL** the ratings that item $I_{10}$ has received from different users, i.e. $A_{1, 10}, A_{2, 10}, A_{3, 10}, ..., A_{n, 10}$.

Out of these, we use all the ratings that are not NULL/empty.
For eg, if $A_{3, 10} = NULL$, it is not included in the optimization problem.

<br>

**Suppose we're trying to find the user vector $B_{3}$**

Similarly, here we plug-in **ALL** ratings that user $U_3$ has given to different products, i.e. $A_{3, 1}, A_{3, 2}, A_{3, 3}, ..., A_{3, m}$.

If $A_{3, 2}, A_{3, m} = NULL$, they are not included in the optimization problem.

![picture](https://drive.google.com/uc?export=view&id=1CEog1d0CGK0Hsw_nn0W4d8hLx8ukz3g8)

<br>

> **Q. Is there any failure/boundary case for this?**

Yes.

Suppose that there is an user, say, $U_{100}$, who hasn't rated even a single item (Cold start or an old user that doesnt rate).

This means that all values of 100th row in A would be empty, $A_{100, 1}, A_{100, 2}, A_{100, 3}, ..., A_{100, m} = NULL$

Then, it'll be impossible to find the user vector (i.e. $B_{100}$) for him.

Similarly, if there's a new item, that has never been rated, even once (cold start), say item $I_{150}$, then $A_{1, 150}, A_{2, 150}, A_{3, 150}, ..., A_{n, 150} = NULL$, and it becomes impossible to find item vector $I_{150}$.

![picture](https://drive.google.com/uc?export=view&id=1Kf0ZZnqlYDIK_WWAz3aq2La90eKiRJ2x)



> **To summarize (Mental Map):**
- **Recommendation System** can be formed through **Matrix Completion**, which can be achieved through **Matrix Factorization**, which utilises **Stochastic Gradient Descent**.

![picture](https://drive.google.com/uc?export=view&id=1qzZNG2re3v-5hSMCzab4BHBNOXYJE2o0)


> **ASSESSMENTS COVERED:-**
- https://www.scaler.com/hire/test/problem/17086/
- https://www.scaler.com/hire/test/problem/19923/
- https://www.scaler.com/hire/test/problem/19925/
- https://www.scaler.com/hire/test/problem/17088/

---

## Principal Component Analysis (PCA)

We've studied about PCA in the last few classes. Let's connect how PCA is related to concept of MF.

Suppose we have our data matrix, containing **standardised data** $X$ with dimensions `n x d`.

Recall that we calculated the **covariance matrix** using X, $S_{d x d}$, which has dimensions `d x d`.

Covariance matrix is a square and symmetrix matrix.

<br>

> **Q. How did we calculate the covariance matrix?**

Using the relation, $S_{dxd} = \frac{X_{dxn}^T.X_{nxd}}{n-1}$

<br>

Before PCA existed, there was this idea called **Eigen Decomposition**, which is a special type of matrix decomposition.

> **Q. How can we decompose our covariance matrix using the concept of eigen decomposition?**

We can write our $S_{dxd}$ as: $S_{dxd} = W_{dxd} . ∧_{dxd} . W_{dxd}^T$

where <br>
$W_{dxd}$ -> Matrix where columns represent the `d` **eigen vectors** ($v_1,v_2, v_3, ..., v_d$)<br>
$∧_{dxd}$ -> Matrix where all diagonal elements are the **eigen values** ($λ_1, λ_2, λ_3, ..., λ_d$), and all the other elements are 0 <br>
$W_{dxd}^T$ -> This is transpose of $W_{dxd}$, so it becomes a matrix where each row represents the transpose of the `d` eigen vectors.

**NOTE:**
- A property of the singular values in $Σ$ is that: $λ_1>=λ_2>=λ_3>=...>=λ_n$.

<br>

> **INSTRUCTOR NOTE:**
- This is actually how PCA is solved internally, i.e. by decomposing the covariance matrix S as $S_{dxd} = W_{dxd} . ∧_{dxd} . W_{dxd}^T$
- During the PCA lecture, it was not discussed how PCA is solved internally.

![picture](https://drive.google.com/uc?export=view&id=1z1h-ac7Ub23u6gOQRlHoBXR2NoliOJf1)

<br>

> **Q. How is PCA different from Matrix Factorization?**

It's not. PCA is just a special type of MF.

Earlier when we performed MF on $A_{ij}$, we did not care what the decomposing factors $B_i$ and $C_j$ look like, as long as they're satisfying the relation $A_{ij} = B_i^T.C_j$

Whereas, in case of PCA, we have constraints on the value of it's decomposed factors.

<br>

> **Q. What are the constraints on decomposed factors in case of PCA?**

Since $W_{dxd}$ consists of eigen vectors, **each column is perpendicular** to other columns.

Conversely, each row in $W_{dxd}^T$ is perpendicular to other rows.

Also, as we've seen $∧_{dxd}$ is a diagonal matrix.

![picture](https://drive.google.com/uc?export=view&id=1LowAgBaXLrR-1vejNwdfYCuWHSStbS6l)

---

## Singular Value Decomposition (SVD)


One drawback of using PCA is that it can only be applied on the covariance matrix (S), which is square and symmetric.

There is another technique which can help in factorising our data matrix, called **Singular Value Decomposition (SVD)**.

SVD doesn't require a square matrix, it can be directly applied on our rectangular data matrix with dimensions `n x d`.


<br>

> **INSTRUCTOR NOTE:**
- We are not going to prove these relations, we're just looking at them

<br>

> **Q. What does SVD formulation look like?**

Here also, we're trying to decompose the data matrix into product of 3 matrices.

$X_{nxd} = U_{nxn}.Σ_{nxd}.V_{dxd}^T$

Assuming that number of data points is greater than number of dimensions, i.e. $n > d$, lets break down each of these factors:-

- $Σ_{nxd}$: diagonal matrix that contain **d singular values**, and the rest of elements are 0. Notice the dimensions, this is a **rectangular** matrix.

- $U_{nxn}$: **Left singular vectors**, i.e. a Square matrix containing the **eigen vectors** of $X_{nxd}.X_{dxn}^T = S'_{nxn}$ (let), stacked along columns
 - **Note:** $X_{nxd}.X_{dxn}^T$ is not the same as covariance matrix $S_{nxn}$, that was $X_{dxn}^T.X_{nxd}$

- $V_{dxd}$: **Right singular values**, i.e. a square matrix containing **eigen vectors** of $X_{dxn}^T.X_{nxd} = S_{nxn}$ covariance matrix, stacked along columns.
 - **Note:** In SVD formulation, $V_{dxd}$ is transposed, therefore, these eigen vectors become stacked along the rows.


> **NOTE:**
- Eigenvectors and Eigenvalues are only defined for squared matrices, whereas singular values and singular vectors are defined for rectangular matrices also.
- Notice that $V_{dxd}$ becomes exactly same as $W_{dxd}$ that we saw in PCA.



<br>

> **Q. How do we find singular values?**

Singular values are related to eigen values as per the following relation:

$S_i^2 = λ_i * (n-1)$

**NOTE:**
- A property of the singular values in $Σ$ is that: $s_1>=s_2>=s_3>=...>=s_n$.

![picture](https://drive.google.com/uc?export=view&id=1fhrFTWGMyBtdXyVcy3md1h1YxzZzqAuh)


![picture](https://drive.google.com/uc?export=view&id=1Ny2ihrbM6bwfdWPBY-QCcOVtOpdNx5Ir)

![picture](https://drive.google.com/uc?export=view&id=1rvuRF9hFyEWftAFKv8LpH79xziHAsAUJ)

> **Q. Why is SVD important for us?**

The concept of SVD is very closely related to PCA. However, there is another concept, called `truncated SVD` that is very important.

We will study about this shortly.

This concept is capable of coming up with very interesting feature engineering, given a data matrix $X_{nxd}$

In the next lecture, we will see other special cases and applications of matrix factorization, and connect all of it to the context of Recommender Systems.

> **ASSESSMENTS COVERED:- (MISCELLANEOUS)**
- https://www.scaler.com/hire/test/problem/20343/
 >> **Coding questions:-**
- https://www.scaler.com/hire/test/problem/24629/

> **SVD:-**
- https://www.scaler.com/hire/test/problem/17090/
- https://www.scaler.com/hire/test/problem/17089/
- https://www.scaler.com/hire/test/problem/26318/

---
---